<a href="https://colab.research.google.com/github/kuds/rl-connect-four/blob/main/%5BConnect%20Four%5D%20AlphaZero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Playing Connect Four with AlphaZero (OpenSpiel)

Trains AlphaZero on `connect_four` using OpenSpiel's reference
implementation, then exports the trained agent under
`$ARTIFACT_DIR/agents/alphazero/` so the tournament notebook in this
repo can pick it up alongside PPO and DQN.

This notebook does **not** use RLlib — RLlib's AlphaZero is deprecated.
OpenSpiel's reference AZ runs MCTS-driven self-play (so there is no
`SelfPlayCallback` league here) and trains a value+policy net via
TensorFlow.

The default config is sized for ~10–15 min on a Colab L4. Bump
`max_steps` and `nn_width` for stronger play; bump `max_simulations` /
`eval_simulations` to trade wall-clock for strength at eval time.

## References:
- [OpenSpiel — AlphaZero](https://github.com/google-deepmind/open_spiel/tree/master/open_spiel/python/algorithms/alpha_zero)
- [OpenSpiel — Documentation](https://openspiel.readthedocs.io/en/latest/intro.html)


In [ ]:
!pip install open_spiel "pettingzoo[classic]" tensorflow imageio


In [ ]:
import json
import math
import multiprocessing as mp
import platform
import shutil
import time
from importlib.metadata import version
from pathlib import Path

import imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pyspiel
from open_spiel.python.algorithms import mcts
from open_spiel.python.algorithms.alpha_zero import alpha_zero as az_train
from open_spiel.python.algorithms.alpha_zero import evaluator as az_evaluator
from open_spiel.python.algorithms.alpha_zero import model as az_model


In [ ]:
print(f"Python Version:    {platform.python_version()}")
print(f"Numpy Version:     {version('numpy')}")
print(f"Open Spiel Version:{version('open_spiel')}")
try:
    print(f"TensorFlow Version:{version('tensorflow')}")
except Exception as e:
    print(f"TensorFlow not importable from metadata: {e!r}")
print(f"CPUs Available:    {mp.cpu_count()}")


In [ ]:
class Args:
    """Demo-grade AlphaZero config sized for a Colab L4.

    Strength scales most directly with ``max_steps`` (training length),
    ``nn_width`` / ``nn_depth`` (capacity), and ``max_simulations`` /
    ``eval_simulations`` (search depth). Bump those to push strength.
    """

    def __init__(self):
        self.env = "connect_four"

        # Where OpenSpiel's AZ writes its run (checkpoints + JSONL logs).
        # Lives on local disk during training; we copy the final
        # checkpoint into the agent dir at the end so it persists.
        self.run_path = "/tmp/alpha_zero_c4"

        # Training-loop sizing — keep small so the notebook finishes on
        # Colab. ``max_steps`` is the number of learner steps; each step
        # consumes a batch from the replay buffer fed by the actor
        # processes.
        self.max_steps = 10
        self.checkpoint_freq = 2

        # Self-play and evaluation actors (separate processes spawned by
        # OpenSpiel's AZ). Tune ``actors`` based on available CPUs;
        # leaving 2 free for the learner + main process is usually safe.
        self.actors = 2
        self.evaluators = 1
        self.evaluation_window = 20
        self.eval_levels = 7

        # MCTS / exploration. Eval-time sims can differ from training to
        # trade tournament wall-clock for strength.
        self.uct_c = 2.0
        self.max_simulations = 50
        self.eval_simulations = 50
        self.policy_alpha = 0.25
        self.policy_epsilon = 0.25
        self.temperature = 1.0
        self.temperature_drop = 10

        # Replay buffer + optimizer.
        self.replay_buffer_size = 2 ** 13
        self.replay_buffer_reuse = 4
        self.train_batch_size = 128
        self.learning_rate = 1e-3
        self.weight_decay = 1e-4

        # Net architecture. ``nn_model`` is one of {"resnet", "mlp", "conv2d"}.
        self.nn_model = "resnet"
        self.nn_width = 64
        self.nn_depth = 3

        # Eval / artifact knobs (used after training).
        self.num_eval_games_per_side = 100
        self.win_rate_threshold = 0.95  # for parity with PPO meta.json


args = Args()


In [ ]:
# Wipe the run dir so re-runs of this cell start fresh.
run_dir = Path(args.run_path)
if run_dir.exists():
    shutil.rmtree(run_dir)
run_dir.mkdir(parents=True, exist_ok=True)

# Build the AZ config. ``observation_shape`` and ``output_size`` are
# inferred from the game by alpha_zero.alpha_zero when set to None.
az_config = az_train.Config(
    game=args.env,
    path=str(run_dir),
    learning_rate=args.learning_rate,
    weight_decay=args.weight_decay,
    train_batch_size=args.train_batch_size,
    replay_buffer_size=args.replay_buffer_size,
    replay_buffer_reuse=args.replay_buffer_reuse,
    max_steps=args.max_steps,
    checkpoint_freq=args.checkpoint_freq,
    actors=args.actors,
    evaluators=args.evaluators,
    evaluation_window=args.evaluation_window,
    eval_levels=args.eval_levels,
    uct_c=args.uct_c,
    max_simulations=args.max_simulations,
    policy_alpha=args.policy_alpha,
    policy_epsilon=args.policy_epsilon,
    temperature=args.temperature,
    temperature_drop=args.temperature_drop,
    nn_model=args.nn_model,
    nn_width=args.nn_width,
    nn_depth=args.nn_depth,
    observation_shape=None,
    output_size=None,
    quiet=True,
)

print("Starting AlphaZero training. This is the slow cell — actor and "
      "evaluator processes self-play games while the learner trains.")
_t0 = time.time()
az_train.alpha_zero(az_config)
wall_clock = time.time() - _t0
print(f"AlphaZero training finished in {wall_clock:.1f}s.")


## Post-training artifacts

### Mount Google Drive

In [ ]:
# Mount Google Drive so models, learning-curve plots, game videos, and the
# results summary persist across Colab runtimes. Set USE_GDRIVE = False to
# keep everything local under ./artifacts/ (e.g. when running outside
# Colab).
USE_GDRIVE = True
GDRIVE_ARTIFACT_SUBDIR = "rl-connect-four"

drive_mounted = False
if USE_GDRIVE:
    try:
        from google.colab import drive  # noqa: WPS433 (Colab-only import)
        drive.mount("/content/drive", force_remount=False)
        drive_mounted = True
    except (ImportError, ModuleNotFoundError):
        print("Not running in Colab — Google Drive unavailable, using ./artifacts/")
    except Exception as e:
        print(f"Could not mount Google Drive ({e!r}); using ./artifacts/")

if drive_mounted:
    ARTIFACT_DIR = Path("/content/drive/MyDrive") / GDRIVE_ARTIFACT_SUBDIR
else:
    ARTIFACT_DIR = Path("artifacts")

AGENT_NAME = "alphazero"
AGENT_DIR = ARTIFACT_DIR / "agents" / AGENT_NAME
VIDEO_DIR = AGENT_DIR / "videos"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
AGENT_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifact directory: {ARTIFACT_DIR}")
print(f"Agent directory:    {AGENT_DIR}")
print(f"Video directory:    {VIDEO_DIR}")


### Load trained model

In [ ]:
# Find the latest checkpoint OpenSpiel's AZ wrote to args.run_path.
#
# AZ writes checkpoints as ``checkpoint-N.{index,data-00000-of-00001}``
# files under ``run_path``. ``Model.from_checkpoint`` wants the path
# *prefix* (no extension), so we strip the trailing extension after
# picking the highest step.
checkpoint_files = list(run_dir.glob("checkpoint-*.index"))
if not checkpoint_files:
    # Older OpenSpiel versions skip the .index suffix.
    checkpoint_files = [
        p for p in run_dir.glob("checkpoint-*")
        if p.is_file() and not p.name.endswith(".meta")
    ]
if not checkpoint_files:
    raise RuntimeError(
        f"No AlphaZero checkpoint files found under {run_dir}. "
        f"Did training complete?"
    )


def _checkpoint_step(p):
    stem = p.name.rsplit(".", 1)[0]  # drop extension
    try:
        return int(stem.rsplit("-", 1)[-1])
    except ValueError:
        return -1


latest_ckpt = max(checkpoint_files, key=_checkpoint_step)
checkpoint_prefix = str(latest_ckpt).rsplit(".", 1)[0]
print(f"Latest checkpoint prefix: {checkpoint_prefix}")

# Load the model + build an MCTS bot for in-notebook eval.
game = pyspiel.load_game(args.env)
trained_model = az_model.Model.from_checkpoint(checkpoint_prefix)
trained_evaluator = az_evaluator.AlphaZeroEvaluator(game, trained_model)
trained_bot = mcts.MCTSBot(
    game=game,
    uct_c=args.uct_c,
    max_simulations=args.eval_simulations,
    evaluator=trained_evaluator,
    solve=False,
    verbose=False,
)
print("Loaded trained model and built MCTS bot for evaluation.")


### Evaluation helpers

In [ ]:
# Eval helpers. AlphaZero needs the full ``pyspiel.State`` (not just
# info_state) for MCTS rollouts, so unlike the PPO/DQN notebooks we
# drive ``pyspiel.State`` directly rather than going through
# ``rl_environment.Environment``.


def az_action(state):
    """MCTS+net action for the trained AlphaZero bot."""
    return int(trained_bot.step(state))


def random_action(state, rng):
    legal = state.legal_actions()
    return int(rng.choice(legal))


def play_match(game, agent_p0, agent_p1, rng):
    """Play one game; return ``state.returns()`` (per-player rewards)."""
    state = game.new_initial_state()
    while not state.is_terminal():
        if state.is_chance_node():
            outcomes, probs = zip(*state.chance_outcomes())
            action = int(rng.choice(outcomes, p=list(probs)))
        else:
            agent = agent_p0 if state.current_player() == 0 else agent_p1
            action = agent(state)
        state.apply_action(action)
    return state.returns()


def evaluate_az(opp_fn, num_games_per_side, seed=0):
    """Symmetric AZ vs opp eval. ``opp_fn(state, rng)`` -> action."""
    rates = {}
    for role, key in [(0, "az_as_p0"), (1, "az_as_p1")]:
        rng = np.random.default_rng(seed + role)
        wins = losses = draws = 0
        for _ in range(num_games_per_side):
            opp_with_rng = (lambda s, _rng=rng: opp_fn(s, _rng))
            if role == 0:
                rewards = play_match(game, az_action, opp_with_rng, rng)
            else:
                rewards = play_match(game, opp_with_rng, az_action, rng)
            r = rewards[role]
            if r > 0:
                wins += 1
            elif r < 0:
                losses += 1
            else:
                draws += 1
        n = num_games_per_side
        rates[key] = {
            "wins": wins, "losses": losses, "draws": draws,
            "win_rate": wins / n, "loss_rate": losses / n,
            "draw_rate": draws / n, "games": n,
        }
    return rates


def overall_win_rate_ci(eval_result):
    wins = sum(side["wins"] for side in eval_result.values())
    n = sum(side["games"] for side in eval_result.values())
    p = wins / n if n else 0.0
    se = math.sqrt(p * (1 - p) / n) if n else 0.0
    return {
        "win_rate": p,
        "ci95_low": p - 1.96 * se,
        "ci95_high": p + 1.96 * se,
        "wins": wins,
        "games": n,
    }


### Learning curves

In [ ]:
# OpenSpiel's AZ writes JSONL logs into the run dir. The learner log
# captures per-step training stats; we plot whatever is available and
# fall back gracefully if the format differs across OpenSpiel versions.
learner_log = run_dir / "learner.jsonl"
log_rows = []
if learner_log.exists():
    with open(learner_log) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                log_rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass

if log_rows:
    log_df = pd.DataFrame(log_rows)
    print(f"Parsed {len(log_df)} learner log rows; columns: {list(log_df.columns)}")
else:
    log_df = pd.DataFrame()
    print(f"No learner log found at {learner_log} — skipping curve plot.")


def _first(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


step_col = _first(log_df, ["step", "global_step"])
loss_col = _first(log_df, ["loss", "total_loss"])
ploss_col = _first(log_df, ["policy_loss"])
vloss_col = _first(log_df, ["value_loss"])
l2_col = _first(log_df, ["l2_loss", "l2reg_loss"])

curves_path = AGENT_DIR / "learning_curves.png"
if step_col is not None and any(c is not None for c in [loss_col, ploss_col, vloss_col]):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    x = log_df[step_col]
    if loss_col is not None:
        axes[0].plot(x, log_df[loss_col], color="#264653")
    axes[0].set_title("Total loss" if loss_col else "Total loss (n/a)")
    axes[0].set_xlabel("learner step")
    axes[0].grid(alpha=0.3)
    if ploss_col is not None:
        axes[1].plot(x, log_df[ploss_col], color="#6a4c93")
    axes[1].set_title("Policy loss" if ploss_col else "Policy loss (n/a)")
    axes[1].set_xlabel("learner step")
    axes[1].grid(alpha=0.3)
    if vloss_col is not None:
        axes[2].plot(x, log_df[vloss_col], color="#f4a261")
    axes[2].set_title("Value loss" if vloss_col else "Value loss (n/a)")
    axes[2].set_xlabel("learner step")
    axes[2].grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(curves_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved {curves_path}")
else:
    print("Not enough columns in learner log to plot a curve.")


### Evaluate vs Random baseline

In [ ]:
# Quantitative evaluation against a uniform-random baseline. Symmetric
# across player-0 / player-1 to account for Connect Four's first-move
# advantage.
random_baseline = evaluate_az(
    random_action, args.num_eval_games_per_side, seed=0,
)
random_summary = overall_win_rate_ci(random_baseline)

print("AlphaZero vs Random baseline:")
for role, stats in random_baseline.items():
    print(
        f"  {role}: win={stats['win_rate']:.3f}  "
        f"loss={stats['loss_rate']:.3f}  draw={stats['draw_rate']:.3f}  "
        f"(n={stats['games']})"
    )
print(
    f"  overall: win_rate={random_summary['win_rate']:.3f} "
    f"(95% CI [{random_summary['ci95_low']:.3f}, "
    f"{random_summary['ci95_high']:.3f}], n={random_summary['games']})"
)


### Record games as GIFs

In [ ]:
# Record games as GIFs using PettingZoo's Connect Four renderer driven
# in lockstep with a pyspiel.State that the AZ MCTS bot reads from.
try:
    from pettingzoo.classic import connect_four_v3
    PETTINGZOO_AVAILABLE = True
except ImportError as e:
    print(f"PettingZoo not installed ({e}); skipping video recording.")
    PETTINGZOO_AVAILABLE = False


def record_game_gif(agent_p0, agent_p1, output_path, seed=0):
    pz_env = connect_four_v3.env(render_mode="rgb_array")
    pz_env.reset(seed=seed)

    state = game.new_initial_state()
    rng = np.random.default_rng(seed)
    frames = [pz_env.render()]
    while not state.is_terminal():
        if state.is_chance_node():
            outcomes, probs = zip(*state.chance_outcomes())
            action = int(rng.choice(outcomes, p=list(probs)))
        else:
            agent = agent_p0 if state.current_player() == 0 else agent_p1
            action = agent(state)
        pz_env.step(action)
        state.apply_action(action)
        frames.append(pz_env.render())
    pz_env.close()

    rewards = state.returns()
    if rewards[0] > 0:
        result = "p0 wins"
    elif rewards[1] > 0:
        result = "p1 wins"
    else:
        result = "draw"
    frames.extend([frames[-1]] * 4)
    imageio.mimsave(str(output_path), frames, fps=1.5, loop=0)
    return {
        "moves": len(frames) - 5,
        "result": result,
        "path": str(output_path),
        "num_frames": len(frames),
    }


recorded_games = {}
if PETTINGZOO_AVAILABLE:
    rng_for_random = np.random.default_rng(123)
    rand_with_rng = (lambda s, _rng=rng_for_random: random_action(s, _rng))
    specs = [
        ("alphazero_vs_random_p0.gif", az_action, rand_with_rng),
        ("alphazero_vs_random_p1.gif", rand_with_rng, az_action),
    ]
    for filename, a0, a1 in specs:
        out = VIDEO_DIR / filename
        info = record_game_gif(a0, a1, out, seed=0)
        recorded_games[filename] = info
        print(f"Saved {out.name}: {info['moves']} moves, {info['result']}")
    print(f"\nAll videos saved to: {VIDEO_DIR}")


### Persist artifacts

In [ ]:
# Copy the AZ checkpoint files into AGENT_DIR/checkpoint/ so they
# survive across Colab runtimes. We copy *all* files belonging to the
# latest checkpoint (the .index + data shards) plus the model graph
# files OpenSpiel writes alongside (typically ``vplus.pbtxt`` /
# ``checkpoint``); harmless to over-copy.
checkpoint_dst = AGENT_DIR / "checkpoint"
if checkpoint_dst.exists():
    shutil.rmtree(checkpoint_dst)
checkpoint_dst.mkdir(parents=True, exist_ok=True)

latest_step = _checkpoint_step(latest_ckpt)
for p in run_dir.iterdir():
    if not p.is_file():
        continue
    # Always copy graph / config files.
    if p.suffix in {".pbtxt", ".json", ".txt"} or p.name == "checkpoint":
        shutil.copy2(p, checkpoint_dst / p.name)
        continue
    # Copy the latest checkpoint's shards.
    name = p.name
    if name.startswith("checkpoint-"):
        try:
            step = int(name.rsplit(".", 1)[0].rsplit("-", 1)[-1])
        except ValueError:
            continue
        if step == latest_step:
            shutil.copy2(p, checkpoint_dst / name)

# Inside AGENT_DIR/checkpoint the prefix that ``Model.from_checkpoint``
# wants is just ``checkpoint-<latest_step>``.
exported_prefix_relative = f"checkpoint-{latest_step}"
print(f"Checkpoint copied to: {checkpoint_dst}")
print(f"Exported checkpoint prefix (relative): {exported_prefix_relative}")

summary = {
    "algo": "AlphaZero",
    "env": args.env,
    "training_steps": args.max_steps,
    "wall_clock_seconds": float(wall_clock),
    "artifact_dir": str(ARTIFACT_DIR),
    "agent_dir": str(AGENT_DIR),
    "drive_mounted": drive_mounted,
    "eval_vs_random": {
        "per_side": random_baseline,
        "overall": random_summary,
    },
    "checkpoint_path": str(checkpoint_dst),
    "checkpoint_prefix_relative": exported_prefix_relative,
    "graphs": {"learning_curves": str(curves_path)},
    "videos": recorded_games,
    "mcts": {
        "uct_c": args.uct_c,
        "max_simulations_train": args.max_simulations,
        "eval_simulations": args.eval_simulations,
    },
}
results_json_path = AGENT_DIR / "results.json"
with open(results_json_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Summary written to: {results_json_path}")
print(json.dumps(summary, indent=2, default=str))


### Export agent for tournament

In [ ]:
# Export this agent for the tournament notebook. We write:
#   - adapter.py   load_agent(agent_dir, env_name) -> act callable
#   - meta.json    algo + training metadata + MCTS settings the adapter
#                  reads at load time
# (The checkpoint already lives under AGENT_DIR/checkpoint from the
# previous cell, with the latest step recorded in
# ``exported_prefix_relative``.)

adapter_src = '''"""Tournament adapter for an OpenSpiel AlphaZero agent.

Loaded by the tournament notebook via importlib. Exposes a single
``load_agent(agent_dir, env_name)`` returning a callable matching:

    act(obs: np.ndarray, legal_actions: list[int], state=None) -> int

AlphaZero requires ``state`` (a ``pyspiel.State``) because MCTS rolls
out simulations from the current state. ``obs`` and ``legal_actions``
are accepted for contract compatibility with the RLlib adapter but
ignored here.
"""

import json
from pathlib import Path

import numpy as np
import pyspiel
from open_spiel.python.algorithms import mcts
from open_spiel.python.algorithms.alpha_zero import evaluator as az_evaluator
from open_spiel.python.algorithms.alpha_zero import model as az_model


def load_agent(agent_dir, env_name):
    agent_dir = Path(agent_dir)
    meta_path = agent_dir / "meta.json"
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}

    eval_simulations = int(meta.get("eval_simulations", 50))
    uct_c = float(meta.get("uct_c", 2.0))
    prefix_relative = meta.get("checkpoint_prefix_relative")

    checkpoint_dir = agent_dir / "checkpoint"
    if prefix_relative:
        checkpoint_prefix = str(checkpoint_dir / prefix_relative)
    else:
        # Fall back to discovering the latest checkpoint inside the dir.
        candidates = list(checkpoint_dir.glob("checkpoint-*.index")) or [
            p for p in checkpoint_dir.glob("checkpoint-*")
            if p.is_file() and p.suffix not in {".meta"}
        ]
        if not candidates:
            raise RuntimeError(
                f"No AlphaZero checkpoint found in {checkpoint_dir}"
            )

        def _step(p):
            stem = p.name.rsplit(".", 1)[0]
            try:
                return int(stem.rsplit("-", 1)[-1])
            except ValueError:
                return -1

        latest = max(candidates, key=_step)
        checkpoint_prefix = str(latest).rsplit(".", 1)[0]

    game = pyspiel.load_game(env_name)
    model = az_model.Model.from_checkpoint(checkpoint_prefix)
    evaluator = az_evaluator.AlphaZeroEvaluator(game, model)
    bot = mcts.MCTSBot(
        game=game,
        uct_c=uct_c,
        max_simulations=eval_simulations,
        evaluator=evaluator,
        solve=False,
        verbose=False,
    )

    def act(obs, legal_actions, state=None):  # noqa: ARG001
        if state is None:
            raise RuntimeError(
                "AlphaZero adapter requires a pyspiel.State (state=...). "
                "The tournament notebook must drive pyspiel.State so it "
                "can pass state into act()."
            )
        return int(bot.step(state))

    return act
'''
adapter_path = AGENT_DIR / "adapter.py"
adapter_path.write_text(adapter_src)
print(f"Wrote adapter to: {adapter_path}")

meta = {
    "agent_name": AGENT_NAME,
    "algo": "AlphaZero",
    "env": args.env,
    "training_steps": args.max_steps,
    "wall_clock_seconds": float(wall_clock),
    "win_rate_threshold": args.win_rate_threshold,
    "framework": "tensorflow",
    "checkpoint_path": str(checkpoint_dst),
    "checkpoint_prefix_relative": exported_prefix_relative,
    "uct_c": args.uct_c,
    "max_simulations_train": args.max_simulations,
    "eval_simulations": args.eval_simulations,
    "nn_model": args.nn_model,
    "nn_width": args.nn_width,
    "nn_depth": args.nn_depth,
    "results_json": str(results_json_path),
}
meta_path = AGENT_DIR / "meta.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2, default=str)
print(f"Wrote meta to: {meta_path}")
